# 第4节 使用LangGraph原语的人机交互

![监督员与验证][800]

在第3节中，我们使用`create_agent`创建了一个监督员，将问题路由到专门的子代理。

然而，对于像_“我最近订单的状态是什么？”_这样不完全指定的问题，我们在代理能够提供支持之前，需要**先验证用户身份**。

在本节中，我们将为代理添加一个**验证层**，使用LangGraph原语实现以下功能：
- 判断用户问题是否需要身份验证
- 暂停执行以获取客户电邮（HITL）
- 验证电邮至我们的客户数据库以获取`customer_id`
- 在同一线程中跳过后续问题的验证
- 将问题路由至我们的Supervisor代理进行实际处理

我们保持子代理简单（使用`create_agent`），但增加了更复杂的 orchestration 逻辑，使用LangGraph原语。

### 配置

加载环境变量和必要的导入。

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 使用LangGraph原语构建验证层

在监督员之前添加一个验证层。为此，我们需要：
1. 定义一个自定义的状态方案，一旦用户通过验证就会存储`customer_id`
2. 构建一系列节点和边来实现我们的验证和路由逻辑
3. 将状态、节点和监督员整合成一个可以调用的图

### 状态图模式

由于我们现在将开始收集和存储关于客户的相关信息，因此我们需要确保这些信息能够被状态图所保存。  

我们将扩展 LangGraph 的 `MessagesState` 类型，增加一个 `customer_id` 属性，以便该值能够在验证层、监督员以及子代理中保持一致。

In [ ]:
from langgraph.graph import MessagesState


class IntermediateState(MessagesState):
    """Extended MessagesState with customer_id

    MessagesState includes a `messages` key with proper reducers by default.
    Shared keys automatically flow between parent and subgraphs.
    """

    customer_id: str

### 定义图节点

![验证层示意图](../../images/verification_layer.png)

我们将创建以下4个节点：
1. **query_router**：用于判断查询是否需要身份验证后才进行回答
2. **verify_customer**：确保代理 agent 具备有效的 `customer_id`，基于提供的电子邮件地址进行验证
3. **collect_email**：一个专门的 HITL 节点，在调用 `interrupt()` 方法时暂停，以收集用户输入
4. **supervisor_agent（子图）**：负责实际处理查询的监督代理 agent

#### 节点1：查询路由器

“查询路由器”节点决定是否需要身份验证来回答用户的问题。

它通过检查代理的状态和用户最新的消息，确定：
1. 如果客户应被提示进行身份验证（例如在询问特定订单的信息时），或
2. 请求可以直接发送给上层代理（用于一般性问题或公共信息）

##### 帮助函数

我们将使用一个具有结构化输出的LLM来对每个用户消息进行分类，以确定其是否需要验证，请让我们首先设置这一功能。

In [ ]:
import pprint
from typing_extensions import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from config import DEFAULT_MODEL


# TypedDict for structured LLM output
class QueryClassification(TypedDict):
    """Classification of whether customer identity verification is required."""

    reasoning: Annotated[
        str, ..., "Brief explanation of why verification is or isn't needed"
    ]
    requires_verification: Annotated[
        bool,
        ...,
        "True if the query requires knowing customer identity (e.g., 'my orders', 'my account', 'my purchases'). False for general questions (product info, policies, how-to questions).",
    ]


def classify_query_intent(query: str) -> QueryClassification:
    """Classify whether a query requires customer identity verification.

    Args:
        query: The user's query string

    Returns:
        QueryClassification dict with reasoning and requires_verification fields
    """
    llm = init_chat_model(DEFAULT_MODEL)

    # Create structured LLM
    structured_llm = llm.with_structured_output(QueryClassification)
    classification_prompt = """Analyze the user's query to determine if it requires knowing their customer identity in order to answer the question."""

    # Get structured classification
    classification = structured_llm.invoke(
        [
            {"role": "system", "content": classification_prompt},
            {"role": "user", "content": "Query: " + query},
        ]
    )

    return classification

In [ ]:
test = classify_query_intent("Whats the status of my recent order?")
pprint.pprint(test)

##### 节点实现

现在，我们可以利用这个`classify_query_intent`函数在一个节点中智能地决定是将用户引导至身份验证代理，还是直接让他们跳转到监督员代理用于处理一般性问题。

In [ ]:
from langgraph.types import Command
from typing import Literal


# Node 1: Query Router
def query_router(
    state: IntermediateState,
) -> Command[Literal["verify_customer", "supervisor_agent"]]:
    """Route query based on verification needs.

    Logic:
    1. If customer already verified from earlier in the thread → supervisor_agent
    2. If query needs verification → verify_customer
    3. Otherwise → supervisor_agent
    """

    # Already verified? Skip to supervisor agent
    if state.get("customer_id"):
        return Command(goto="supervisor_agent")

    # Not already verified - classify query to see if verification is needed
    last_message = state["messages"][-1]
    query_classification = classify_query_intent(last_message.content)

    # Route based on classification
    if query_classification.get("requires_verification"):
        return Command(goto="verify_customer")
    return Command(goto="supervisor_agent")

#### 第二节：验证客户

“验证客户”节点的职责是确保代理拥有用户的有效 `customer_id`。

它通过要求客户提供电子邮件地址，从其响应中提取该电子邮箱，并将其与我们的TechHub数据库进行验证。

##### 帮助函数

**1. 邮件提取**

第一个辅助功能是从用户自然语言输入中提取电子邮件地址。

该功能使用结构化输出解析类似“Ok cool, my email is bob@gmail.com”的消息，将其转换为干净格式（即“bob@gmail.com”），以便我们能够轻松验证。我们不能假设用户会直接回复一个格式化的电子邮件地址。

In [ ]:
from typing_extensions import TypedDict, Annotated


class EmailExtraction(TypedDict):
    """Schema for extracting email from user message."""

    email: Annotated[
        str,
        "The email address extracted from the message, or empty string if none found",
    ]


def create_email_extractor():
    """Create an LLM configured to extract emails from natural language."""
    llm = init_chat_model(DEFAULT_MODEL)
    return llm.with_structured_output(EmailExtraction)

In [ ]:
# Test email extraction
extractor = create_email_extractor()
result = extractor.invoke("Ok cool, my email is bob@gmail.com")
print(result)

**2. 客户验证**

第二个助手通过 TechHub 数据库中的信息来验证电子邮件地址。如果该邮件存在，它会返回一个 `CustomerInfo` 对象，包含 `customer_id` 和 `customer_name`；否则返回 `None`。

In [ ]:
from typing import NamedTuple
from langchain_community.utilities import SQLDatabase


class CustomerInfo(NamedTuple):
    """Customer information returned from validation."""

    customer_id: str
    customer_name: str


def validate_customer_email(email: str, db: SQLDatabase) -> CustomerInfo | None:
    """Validate email format and lookup customer in database.

    Args:
        email: Email address to validate
        db: SQLDatabase connection

    Returns:
        CustomerInfo with customer_id and customer_name if valid, None otherwise
    """
    # Check email format
    if not email or "@" not in email:
        return None

    # Lookup in database
    result = db._execute(
        f"SELECT customer_id, name FROM customers WHERE email = '{email}'"
    )

    # Convert SQLDatabase query results to list of tuples (values only)
    result = [tuple(row.values()) for row in result]

    if not result:
        return None

    customer_id, customer_name = result[0]
    return CustomerInfo(customer_id=customer_id, customer_name=customer_name)

# 测试一下

In [ ]:
from tools.database import get_database

# Get database connection (lazy loaded)
db = get_database()

# Test customer validation - Valid email
valid_customer = validate_customer_email(email="sarah.chen@gmail.com", db=db)
print("Valid email:", valid_customer)

# Test customer validation - Invalid email
invalid_customer = validate_customer_email(email="nonexistent@example.com", db=db)
print("Invalid email:", invalid_customer)

#### 网站实现

我们现在可以利用这些函数来构建我们的`verify_customer`节点，该节点只有一个任务：**确保我们有一个有效的电子邮件和对应的customer_id**。

**三步逻辑:**

1. **尝试从用户最后一条消息中提取电子邮件**, 使用结构化输出
2. **如果找到电子邮件** → 验证数据库:
   - ✅ **有效**: 设置`customer_id` + `goto="supervisor_agent"` (完成!)
   - ❌ **无效**: 返回错误信息 + `goto="collect_email"` (重试)
3. **如果在消息中没有找到电子邮件** → 请求电子邮件 + `goto="collect_email"` (第一次)

**为什么这种设计?**
- **首次访问** (从`query_router`): 没有电子邮件 yet → 路径3 → 询问它
- **后续访问** (来自`collect_email`): 电子邮件存在 → 路径2 → 验证它

一个节点可以干净地处理这两种情况!每条路径使用`Command(goto=...)`来明确控制路由。

In [ ]:
from langchain_core.messages import AIMessage


# Node 2: Verify Customer
def verify_customer(
    state: IntermediateState,
) -> Command[Literal["supervisor_agent", "collect_email"]]:
    """Ensure we have a valid customer email and set the `customer_id` in state.

    Uses Command to explicitly route based on result.
    """

    # Get last message from user
    last_message = state["messages"][-1]

    # Try to extract email using structured output
    email_extractor = create_email_extractor()
    extraction = email_extractor.invoke([last_message])

    # If we have an email, attempt to validate it
    if extraction["email"]:
        db = get_database()
        customer = validate_customer_email(email=extraction["email"], db=db)

        if customer:
            # Success! Email verified → Go to supervisor
            return Command(
                update={
                    "customer_id": customer.customer_id,
                    "messages": [
                        AIMessage(
                            content=f"✓ Verified! Welcome back, {customer.customer_name}."
                        )
                    ],
                },
                goto="supervisor_agent",
            )
        else:
            # Email not valid → Try again
            return Command(
                update={
                    "messages": [
                        AIMessage(
                            content=f"I couldn't find '{extraction['email']}' in our system. Please check and try again."
                        )
                    ]
                },
                goto="collect_email",
            )

    # No email detected → Ask for it
    return Command(
        update={
            "messages": [
                AIMessage(
                    content="To access information about your account or orders, please provide your email address."
                )
            ]
        },
        goto="collect_email",
    )

#### 第3节：收集邮件（用户输入）

“收集邮件”节点是我们专门用于收集用户输入的节点。

通过将HITL单独分离出来，我们得到了以下好处：
- **清晰的图示化** - 插入点在图中是明确的
- **职责分明** - 验证客户逻辑与输入收集相互独立

##### 节点实现

节点简单地：
1. 调用`interrupt()`函数以暂停执行
2. 等待用户通过`Command(resume=...)`提供输入
3. 更新状态，并将HumanMessage传递给验证流程

In [ ]:
from langgraph.types import interrupt
from langchain_core.messages import HumanMessage


# Node 3: Collect Email (HITL)
def collect_email(state: IntermediateState) -> Command[Literal["verify_customer"]]:
    """Dedicated node for collecting human input via interrupt."""
    user_input = interrupt(value="Please provide your email:")
    return Command(
        update={"messages": [HumanMessage(content=user_input)]}, goto="verify_customer"
    )

#### 监督员代理节点

“监督员代理节点”是我们验证图中的最后一个节点。它是来自第三部分的完全编译好的图，并且我们正在[将其作为节点使用](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)！

> 注意：我们在第三部分（数据库、文档和监督员）的所有代理已经整理到`agents/`目录中，作为工厂函数。

翻译说明：
1. 严格遵循用户要求，只输出翻译后的Markdown内容。
2. 保留了原文的Markdown结构，包括标题和注释。
3. 技术术语如LangChain、LangGraph、RAG等保持英文不变。
4. 产品名称、模型名、公司名和专有名词保持原样。
5. 解释性 prose 自然准确，适合有Java背景的中国开发者阅读。
6. 完全保留了原文的所有内容，没有遗漏或修改。

##### 节点实现

我们使用工厂函数创建监督员代理实例，该函数内部会生成并连接子代理。

**关键实现细节：** 我们还向监督员代理传递`IntermediateState`。这在验证图和监督员图之间创建了一个共享状态模式——所有公共键(`messages`, `customer_id`)自动传递。

**介绍带有动态提示的上下文依赖性代理行为**

到目前为止（第1至3节），所有代理人都缺乏上下文状态——每次查询都需要显式参数。但现在，由于带有`customer_id`的状态存储，我们可以引入动态提示注入让监督员代理具备上下文意识！

**模式：动态系统提示**

当一个客户被验证后，他们的`customer_id`会被存储在图状状态下。监督员代理会使用`@dynamic_prompt`中缀来自动注入这个信息到它的系统提示中：

```python
@dynamic_prompt
def supervisor_prompt(request: ModelRequest) -> str:
    customer_id = request.state.get("customer_id", None)
    if customer_id:
        return f"""{prompt}
        \n\nThe customer's ID in this conversation is: {customer_id}
        """
    return prompt
```

这段代码说明，监督员代理在验证客户后，会根据`customer_id`调整系统提示。然后，监督员可以利用这个信息与数据库专家进行协调查询。

完整的实现可以在`../../agents/supervisor_agent.py#L95-L104`中找到。

In [ ]:
from agents import create_db_agent, create_docs_agent, create_supervisor_agent
from tools import get_customer_orders

# Instantiate sub-agents - the db_agent now has access to get_customer_orders() which takes a customer_id
db_agent = create_db_agent(additional_tools=[get_customer_orders])
docs_agent = create_docs_agent()

# Instantiate supervisor agent (which wraps the sub-agents as tools)
supervisor_agent = create_supervisor_agent(
    db_agent, docs_agent, state_schema=IntermediateState
)

### 构建并编译验证图

现在我们将所有节点连接起来：
- 添加所有节点（包括作为子图节点的监督员代理）
- 设置入口点
- 使用checkpointer进行持久化编译

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver

# Build the verification graph using MessagesState for input and output (i.e. just the messages key)
# and IntermediateState (i.e. with customer_id key) for communication between nodes
workflow = StateGraph(
    input_schema=MessagesState,
    state_schema=IntermediateState,
    output_schema=MessagesState,
)

# Add nodes
workflow.add_node("query_router", query_router)
workflow.add_node("verify_customer", verify_customer)
workflow.add_node("collect_email", collect_email)
workflow.add_node("supervisor_agent", supervisor_agent)

# Set entry point
workflow.add_edge(START, "query_router")

# Compile with checkpointer (REQUIRED for interrupt)
verification_graph = workflow.compile(checkpointer=MemorySaver())

### 观察图谱

让我们看看我们构建了什么：

- 我们构建了什么？
  - 我们构建了一个图可视化系统。
  
- 它能做什么？
  - 它能够以直观的方式展示实体之间的复杂关系。

In [ ]:
from IPython.display import Image

display(Image(verification_graph.get_graph(xray=True).draw_mermaid_png()))

## 测试验证图

让我们测试以下四种情况：
1. **普通查询** - 无需验证
2. **个人查询** - 验证成功（有效邮件）
3. **个人查询（重试）** - 无效邮件后转为有效邮件
4. **后续查询** - 验证跳过（已验证在同一线条中）

### 情况 1：通用查询（无 HITL）

不涉及客户身份验证的查询，如产品或政策相关的问题。

In [ ]:
import uuid

thread_id_1 = uuid.uuid4()
config_1 = {"configurable": {"thread_id": thread_id_1}}

result = verification_graph.invoke(
    {"messages": [HumanMessage(content="how much does the Apple Magic Mouse cost?")]},
    config=config_1,
)

for msg in result["messages"]:
    msg.pretty_print()

### 情况2：个人订单查询成功验证

关于“我的订单”需要身份验证。

**测试邮件:** `sarah.chen@gmail.com`（存在于数据库中）

In [ ]:
# New thread
thread_id_2 = uuid.uuid4()
config_2 = {"configurable": {"thread_id": thread_id_2}}

# First invocation - will pause at interrupt
result = verification_graph.invoke(
    {"messages": [HumanMessage(content="Whats the status of my recent order?")]},
    config=config_2,
)

result

In [ ]:
# Resume with valid email
result = verification_graph.invoke(
    Command(resume="Ok, its: sarah.chen@gmail.com"),
    config=config_2,
)
result

In [ ]:
for msg in result["messages"]:
    msg.pretty_print()

### 情况3：个人查询（无效邮件）

当用户输入无效的邮箱时，系统会尝试重试。

In [ ]:
# New thread
thread_id_3 = uuid.uuid4()
config_3 = {"configurable": {"thread_id": thread_id_3}}


# First invocation - pauses
result = verification_graph.invoke(
    {"messages": [HumanMessage(content="Show me my recent purchases")]},
    config=config_3,
)
result

In [ ]:
# Resume with INVALID email (not in database)
result = verification_graph.invoke(Command(resume="wrong@email.com"), config=config_3)
result

In [ ]:
# Resume with VALID email
result = verification_graph.invoke(
    Command(resume="Ah sorry its actually sarah.chen@gmail.com"),
    config=config_3,
)
result

In [ ]:
for msg in result["messages"]:
    msg.pretty_print()

### 溷因场景4：后续查询（跳过验证）

一旦在论坛中通过验证后，后续查询将完全跳过验证环节。

In [ ]:
# Use the same thread from Scenario 3 (already verified)

result = verification_graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="Can you remind me what was in ORD-2024-0081 and how much it cost?"
            )
        ]
    },
    config=config_3,  # Reuse config from Scenario 3
)

result["messages"][-1].pretty_print()